### Imports

In [18]:
from pathlib import Path
import sys

In [19]:

root_dir = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(root_dir / "src"))
root_dir


PosixPath('/Volumes/256 SSD/deal_hunter')

In [20]:
from deal_hunter.config import settings

In [21]:
settings.rss_feed_url

['https://www.dealnews.com/c142/Electronics/?rss=1',
 'https://www.dealnews.com/c39/Computers/?rss=1',
 'https://www.dealnews.com/f1912/Smart-Home/?rss=1']

### Testing deals.py , pydantic models

In [22]:
from deal_hunter.models import ScrapedDeal, Deal,DealSelection, Opportunity

In [23]:
#making class
sd = ScrapedDeal(
    title = "Test Product" * 20,
    summary = "test SUmmary",
    url = "https://example.com",
    details = "some details" * 100,
    features = " Feature List" * 100
)



In [24]:
#checking methods
sd.truncate()
print(repr(sd))
print(sd.describe())
print(len(sd.title))
print(len(sd.details))

<Test ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest>
Title: Test ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest
Details: some detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome det
Features: Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature L

In [25]:
# Verifying other models
d = Deal(product_description="A widget", price=29.99, url="https://example.com")
ds = DealSelection(deals=[d])
o = Opportunity(deal=d, estimate=50.0, discount=0.40)
print(o)

deal=Deal(product_description='A widget', price=29.99, url='https://example.com') estimate=50.0 discount=0.4


In [26]:
from deal_hunter.services import Rss_Service

rss = Rss_Service()
deals = rss.scrape_feeds(
    feed_urls=settings.rss_feed_url,
    max_per_feed=3,
)
print(f"Scraped {len(deals)} deals")
for d in deals[:3]:
    print(d.describe())
    print()

Scraped 9 deals
Title: Armitron Connect Link Smartwatch for $32 + free shipping
Details: more


Amazon offers the Armitron Connect Link Smartwatch for $31.89. It's the best price we found by $4. Shipping is free.  
Buy Now at Amazon
Features: tracks heart rate, calories, sleep, menstrual cycle and more
 
step counter, various sport modes, and reminders
 
make and receive calls, receive message notifications and save your favorite contacts
 
free Armitron Connect app compatible with Android 5.0+ and iOS 9.0+
 
Model: 42/1021BKBK
URL:https://www.dealnews.com/products/Armitron/Armitron-Connect-Link-Smartwatch/498308.html?iref=rss-c142

Title: BN-LINK Outdoor 6-Foot Extension Cord Stake with 6 Outlets for $18 + free shipping w/ $35
Details: more


This BN-Link extension cord stake is marked down $12 today.  You'd pay at least a few bucks more elsewhere.  Shipping is free with orders of $35 or more. 
Buy Now at Walmart
Features: 6-ft. cable
 
6 grounded outlets
 
built-in overload protectio

In [27]:
already_seen = {d.url for d in deals}
print(f"Known URLs: {len(already_seen)}")

deals_round2 = rss.scrape_feeds(
    feed_urls=settings.rss_feed_url,
    known_urls=already_seen,
    max_per_feed=3,
)
print(f"Round 2: {len(deals_round2)} new deals (should be 0 or very few)")

Known URLs: 9
Round 2: 0 new deals (should be 0 or very few)


### testing notification service

In [28]:
from deal_hunter.config import settings
from deal_hunter.services.notifications import PushoverNotifier

notifier = PushoverNotifier(
    user_key=settings.pushover_user,
    token=settings.pushover_token,
    url=settings.pushover_url,
)

print("user set?", bool(settings.pushover_user))
print("token set?", bool(settings.pushover_token))
print("url:", settings.pushover_url)

user set? True
token set? True
url: https://api.pushover.net/1/messages.json


In [29]:
ok = notifier.send("Phase 4 test from scanning.ipynb")
print("send() returned:", ok)

send() returned: True


In [30]:
from deal_hunter.config import settings

print("user set?", bool(settings.pushover_user), "len:", len(settings.pushover_user))
print("token set?", bool(settings.pushover_token), "len:", len(settings.pushover_token))
print("url:", settings.pushover_url)


user set? True len: 30
token set? True len: 30
url: https://api.pushover.net/1/messages.json


In [31]:
import requests
from deal_hunter.config import settings

payload = {
    "user": settings.pushover_user,
    "token": settings.pushover_token,
    "message": "debug test from notebook",
    "sound": "cashregister",
}

resp = requests.post(settings.pushover_url, data=payload, timeout=10)
print("HTTP status:", resp.status_code)
print("Response text:", resp.text)

HTTP status: 200
Response text: {"status":1,"request":"7f95f788-39fd-4d6e-966e-1682c9d3184f"}


### testing scanner agent

In [32]:
from deal_hunter.agents.scanner import ScannerAgent

In [33]:
agent = ScannerAgent()
result = agent.test_scan()
print(result)

deals=[Deal(product_description='The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers 3840x2160 resolution, Dolby Vision HDR, and HDR10 support. It uses Roku OS for easy streaming app access and voice assistant compatibility. The TV also includes three HDMI ports for connecting multiple devices.', price=178.0, url='https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142'), Deal(product_description='The Poly Studio P21 is a 21.5-inch 1080p personal meeting display designed for remote work. It includes a built-in 1080p webcam, stereo speakers, privacy shutter, and adjustable lighting features. It also supports wireless charging for compatible devices.', price=30.0, url='https://www.dealnews.com/products/Poly-Studio-P21-21-5-1080-p-LED-Personal-Meeting-Display/378335.html?iref=rss-c39'), Deal(product_description='The Lenovo IdeaPad Slim 5 features an AMD Ryzen 5 8645HS processor, 16GB RAM, and 512GB 

In [35]:
from deal_hunter.agents import MessagingAgent

messenger = MessagingAgent()
print("notifier type:", type(messenger.notifier).__name__)
print("model:", messenger.model)

notifier type: PushoverNotifier
model: gpt-4o-mini


In [36]:
messenger.push("Phase 6 test from scanning.ipynb")

True

In [37]:
messenger.notify(
    "Samsung 60-inch LED TV, 4K UHD, smart platform with built-in apps",
    300,
    1000,
    "https://www.samsung.com/tv",
)

True

### Full pipeline: scan -> notify

Wire the ScannerAgent output into MessagingAgent.
The scanner picks the best deals from RSS, then the messenger crafts an LLM-written notification for each one and pushes it.

In [38]:
from deal_hunter.agents import ScannerAgent, MessagingAgent

scanner = ScannerAgent()
messenger = MessagingAgent()

result = scanner.scan()

if result is None:
    print("No deals found, nothing to notify about.")
else:
    print(f"Scanner returned {len(result.deals)} deals\n")
    for deal in result.deals:
        print(f"  {deal.product_description[:60]}...  ${deal.price}")
        print(f"  {deal.url}\n")

Scanner returned 5 deals

  Armitron Connect Link Smartwatch (model 42/1021BKBK) is a we...  $31.89
  https://www.dealnews.com/products/Armitron/Armitron-Connect-Link-Smartwatch/498308.html?iref=rss-c142

  Refurbished Apple Watch Series 8 GPS 45mm is a smartwatch wi...  $140.0
  https://www.dealnews.com/Refurb-Apple-Watch-Series-8-GPS-45-mm-Smart-Watch-for-140-free-shipping/21822174.html?iref=rss-c142

  Refurb Acer Nitro 34" Ultrawide Curved Monitor features a 34...  $171.0
  https://www.dealnews.com/Refurb-Acer-Nitro-34-Ultrawide-144-p-HDR10-200-Hz-Curved-Monitor-for-171-free-shipping/21822113.html?iref=rss-c39

  Kamrui P2 mini PC is powered by a Ryzen R2544 processor (up ...  $309.0
  https://www.dealnews.com/Kamrui-P2-16-GB-RAM-512-GB-SSD-Mini-PC-for-309-free-shipping/21822021.html?iref=rss-c39

  Shark Matrix Plus 2-in-1 Robot Vacuum & Mop (model AV2620WA)...  $298.99
  https://www.dealnews.com/products/Shark/Shark-Matrix-Plus-2-in-1-Robot-Vacuum-and-Mop-with-Sonic-Mopping/49584

In [39]:
if result and result.deals:
    deal = result.deals[0]
    ok = messenger.notify(
        description=deal.product_description,
        deal_price=deal.price,
        estimated_true_value=deal.price * 2,
        url=deal.url,
    )
    print(f"Push sent: {ok}")
else:
    print("No deals to notify about.")

Push sent: True


### Using test_scan data (no network needed)

If you want to run the pipeline without hitting RSS or OpenAI, use `test_scan()` which returns hardcoded deals.

In [40]:
test_result = scanner.test_scan()

for deal in test_result.deals:
    ok = messenger.notify(
        description=deal.product_description,
        deal_price=deal.price,
        estimated_true_value=deal.price * 2,
        url=deal.url,
    )
    print(f"${deal.price:>7.2f} | pushed={ok} | {deal.product_description[:50]}...")

$ 178.00 | pushed=True | The Hisense R6 Series 55R6030N is a 55-inch 4K UHD...
$  30.00 | pushed=True | The Poly Studio P21 is a 21.5-inch 1080p personal ...
$ 446.00 | pushed=True | The Lenovo IdeaPad Slim 5 features an AMD Ryzen 5 ...
$ 650.00 | pushed=True | The Dell G15 gaming laptop includes an AMD Ryzen 5...


### PlanningAgent smoke test

Wire the scanner, ensemble, and messenger together with a real Chroma collection and run one plan cycle end to end. Uses `test_scan` data by stubbing the scanner so we don't hit OpenAI on the scanning step.

In [ ]:
import chromadb
from deal_hunter.config import settings
from deal_hunter.agents.planner import PlanningAgent

client = chromadb.PersistentClient(path=str(root_dir / "notebooks" / settings.vectorstore_path))
collection = client.get_or_create_collection(settings.vectorstore_collection)
print("collection size:", collection.count())

In [ ]:
from deal_hunter.agents.scanner import ScannerAgent

class StubScanner(ScannerAgent):
    def scan(self, memory=None):
        return self.test_scan()

planner = PlanningAgent(collection, scanner=StubScanner())
best = planner.plan(memory=[])
print("best opportunity:", best)